In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="vllm-api-key"
)


In [2]:
completion = client.chat.completions.create(
    model="meta-llama/Llama-3.2-3B-Instruct",
    messages=[{"role": "user", "content": "What's your name?"}]
)

print(completion.choices[0].message.content)


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


In [12]:
from tqdm.auto import tqdm


class Args:
    def __init__(self, data_file, model_name, client, batch_size):
        self.data_file = data_file
        self.model_name = model_name
        self.client = client
        self.batch_size = batch_size

args = Args(
    data_file="../../prompt_engg/val_data/MSR_BeaverTails_4x56_subset.json",
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    client="vllm",
    batch_size=250
)

In [13]:
from data_generator import data_generator
from model_client import batch_call_litellm
import os
import json


In [ ]:
results = []

i = 0
for batch in tqdm(data_generator(args.data_file, args.batch_size)):
    batch_messages = [[{"role": "user", "content": d['prompt']}] for d in batch]
    responses = batch_call_litellm(batch_messages, model=args.model_name, client=args.client, max_workers=args.batch_size)
    for data, response in zip(batch, responses):
        results.append({**data, f'response_{args.model_name.split("/")[-1]}': response})
    
    i += 1
    if i > 1:
        break

0it [00:00, ?it/s]Process ForkPoolWorker-280:
Process ForkPoolWorker-279:
Process ForkPoolWorker-277:
Process ForkPoolWorker-278:
Process ForkPoolWorker-276:
Traceback (most recent call last):
  File "/home/jamesdin/miniconda3/envs/llm/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Process ForkPoolWorker-275:
  File "/home/jamesdin/miniconda3/envs/llm/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/jamesdin/miniconda3/envs/llm/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
Process ForkPoolWorker-273:
Process ForkPoolWorker-253:
Process ForkPoolWorker-270:
Process ForkPoolWorker-274:
Traceback (most recent call last):
Process ForkPoolWorker-272:
Process ForkPoolWorker-265:
Process ForkPoolWorker-264:
Process ForkPoolWorker-254:
Process ForkPoolWorker-269:
  File "/home/jamesdin/mi

In [11]:
output_file = f"../outputs/results_{args.model_name.split('/')[-1]}_{args.data_file.split('.')[0]}.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, 'w') as f:
    for res in results:
        f.write(json.dumps(res) + '\n')
print(f"Results saved to {output_file}")



Results saved to ../outputs/results_Llama-3.2-3B-Instruct_.json
